# ICH M11 Protocol Draft Generation Demo

This notebook demonstrates how structured clinical trial metadata, such as records derived from the SDTM Trial Summary (TS) domain, can be used with an LLM to generate draft sections of an ICH M11-style clinical trial protocol.

The goal is to show a possible workflow for:

- preparing structured trial metadata as model input
- generating protocol-ready JSON using a structured output schema
- converting generated protocol content into Markdown
- preparing the output for downstream rendering, such as PDF, DOCX, or HTML

## Important Disclaimer

This notebook is intended for demonstration, prototyping, and educational purposes only.

The code, prompts, schemas, and generated outputs are **not production-ready** and should **not** be used as-is for regulatory submissions, clinical trial execution, medical decision-making, quality decisions, or GxP-controlled processes.

The generated protocol content is synthetic or draft material and may contain omissions, inaccuracies, unsupported assumptions, or hallucinated content. All outputs must be reviewed and approved by qualified clinical, medical, statistical, regulatory, legal, and quality professionals before any real-world use.

Use this notebook and any generated outputs **at your own risk**.

## Scope

This example focuses on generating ICH M11-style protocol draft content from structured metadata. It does not replace:

- sponsor protocol authoring processes
- medical writing review
- clinical science review
- statistical review
- regulatory review
- privacy, security, or compliance validation
- formal software validation or computer system validation

## Expected Inputs

The notebook assumes access to structured trial metadata, for example:

- SDTM TS-style records
- optional sponsor metadata
- optional trial design metadata from TA, TE, TV, or TI domains

## Expected Outputs

The workflow produces structured JSON and Markdown that can be used as intermediate artifacts for document generation, review workflows, or synthetic data demonstrations.

In [0]:
# OUTPUT parameters: where to save the generated output
catalog_name = "sandbox"
schema_name = "hfsl"
vol_path = f"/Volumes/{catalog_name}/{schema_name}/ich_m11_protocols"

# INPUT: where to query SDTM Trial Summary (input for the Protocol generation)
# Expected: a TS (Trial Summary)
SDTM_SCHEMA = "sdtm_modeling.foundata_dummy_study_sdtm"
SDTM_TS_TABLE = "ts"
SDTM_TV_TABLE = "tv"

# Study IDs : add here if you want to generate for specific studies only
# To generate protocols for all studies, set STUDY_IDS to None (STUDY_IDS = None)
STUDY_IDS = ["FNN0001-1001", "FNN0001-2005", "FNN0001-2007", "FNN0001-2003", "FNN0001-3001"]

# Protocol overwrite toggle
# If False, studies that already have a PDF in the output volume will be skipped
# Set to True to regenerate all protocols regardless of existing files
PROTOCOL_OVERWRITE = False

# Volume containing fact sheets to augment the Protocol generation
SDTM_FACT_SHEETS = f"/Volumes/{catalog_name}/{schema_name}/m11_input_fact_sheet"

# Validate access to table and studies
for tbl in [SDTM_TS_TABLE, SDTM_TV_TABLE]:
    if not spark.catalog.tableExists(f"{SDTM_SCHEMA}.{tbl}"):
        raise Exception(f"❌ Table {SDTM_SCHEMA}.{tbl} does not exist.")
    else:
        print(f"✅ Table {SDTM_SCHEMA}.{tbl} exists")

# Validate fact sheet volume
try:
    dbutils.fs.ls(SDTM_FACT_SHEETS)
    print(f"✅ Volume {SDTM_FACT_SHEETS} exists")
except Exception:
    raise Exception(f"❌ Volume {SDTM_FACT_SHEETS} does not exist.")

# Validate STUDY_IDS records in both TS and TV tables
if STUDY_IDS is not None:
    for study_id in STUDY_IDS:
        ts_count = spark.sql(f"SELECT COUNT(*) FROM {SDTM_SCHEMA}.{SDTM_TS_TABLE} WHERE STUDYID = '{study_id}'").collect()[0][0]
        tv_count = spark.sql(f"SELECT COUNT(*) FROM {SDTM_SCHEMA}.{SDTM_TV_TABLE} WHERE STUDYID = '{study_id}'").collect()[0][0]
        if ts_count == 0 and tv_count == 0:
            raise Exception(f"❌ Study {study_id} not found in either {SDTM_SCHEMA}.{SDTM_TS_TABLE} or {SDTM_SCHEMA}.{SDTM_TV_TABLE}.")
        elif ts_count == 0:
            print(f"⚠️ Study {study_id} not found in {SDTM_SCHEMA}.{SDTM_TS_TABLE}, but found in {SDTM_SCHEMA}.{SDTM_TV_TABLE}.")
        elif tv_count == 0:
            print(f"⚠️ Study {study_id} found in {SDTM_SCHEMA}.{SDTM_TS_TABLE}, but not in {SDTM_SCHEMA}.{SDTM_TV_TABLE}.")
        else:
            print(f"🔎 Study {study_id} found in both {SDTM_SCHEMA}.{SDTM_TS_TABLE} and {SDTM_SCHEMA}.{SDTM_TV_TABLE} ✅")


In [0]:
vol_name = vol_path.replace("/Volumes/","").replace("/",'.')

# Create schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
print(f"✅ Schema checked/created: {catalog_name}.{schema_name}")

# Create volume if not exists
spark.sql(f"CREATE VOLUME IF NOT EXISTS IDENTIFIER('{vol_name}')")
print(f"✅ Volume checked/created: {vol_name}")

# Set current schema
spark.sql(f"USE {catalog_name}.{schema_name}")
print(f"✅ Using schema: {catalog_name}.{schema_name}")

# Set to True to save the LLM output as Delta table
SAVE_AS_DELTA = True


In [0]:
# Load prompts from external files
with open("assets/user_message_template.txt") as f:
    user_message_template = f.read()

with open("assets/system_message.txt") as f:
    system_message = f.read()

print(f"✅ Loaded user_message_template ({len(user_message_template)} chars)")
print(f"✅ Loaded system_message ({len(system_message)} chars)")

In [0]:
# Extract fact sheets documents from volumes
import os
from markitdown import MarkItDown

study_fact_sheets = {}
for doc in os.listdir(SDTM_FACT_SHEETS):
  #fetch study ID
  study_id = doc.split("_")[0]
  doc_path = os.path.join(SDTM_FACT_SHEETS,doc)   
  md = MarkItDown(enable_plugins=False)
  result = md.convert(doc_path)
  study_fact_sheets[study_id] = result.text_content

display(study_fact_sheets)

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F

# Priority TS parameters for ICH M11 protocol generation
PRIORITY_TS_PARAMS = [
    "ACTSUB",    # Actual Number of Subjects
    "AGEMAX",    # Planned Maximum Age of Subjects
    "AGEMIN",    # Planned Minimum Age of Subjects
    "DCUTDESC",  # Data Cutoff Description
    "DCUTDTC",   # Data Cutoff Date
    "FCNTRY",    # Planned Country of Investigational Sites
    "SENDTC",    # Study End Date
    "SEXPOP",    # Sex of Participants
    "SSTDTC",    # Study Start Date
    "STYPE",     # Study Type
    "INDIC",     # Trial Disease/Condition Indication
    "NARMS",     # Planned Number of Arms
    "THERAREA",  # Therapeutic Area
    "TITLE",     # Trial Title
    "TPHASE",    # Trial Phase Classification
    "TRT",       # Investigational Therapy or Treatment
]


TS_CONTEXT_SECTIONS = [
    "title_page",
    "protocol_synopsis",
    "overall_design",
    "trial_design",
    "trial_population_summary",
    "4.4 Start of Trial and End of Trial",
]


TV_CONTEXT_SECTIONS = [
    "1.2 Trial Schema",
    "1.3 Schedule of Activities",
    "4.1 Description of Trial Design",
    "8 Trial Assessments and Procedures",
]


def rows_to_dicts(df) -> list[dict]:
    """
    Spark-native conversion to list[dict] for compact LLM payloads.
    """
    rows = df.collect()
    return [row.asDict() for row in rows]


def add_record_ref(records: list[dict], source_domain: str) -> list[dict]:
    """
    Add traceability handles so the LLM can cite individual source rows.
    """
    enriched = []

    for i, row in enumerate(records, start=1):
        study_id = row.get("STUDYID") or "UNKNOWN_STUDY"

        if source_domain == "SDTM TS":
            seq = row.get("TSSEQ") or i
            key = row.get("TSPARMCD") or row.get("TSPARM") or "TS"
            record_ref = f"SDTM_TS:{study_id}:{key}:{seq}"

        elif source_domain == "SDTM TV":
            seq = row.get("TVSEQ") or i
            key = row.get("VISITNUM") or row.get("VISIT") or row.get("EPOCH") or "TV"
            record_ref = f"SDTM_TV:{study_id}:{key}:{seq}"

        else:
            record_ref = f"{source_domain.replace(' ', '_')}:{study_id}:{i}"

        enriched_row = dict(row)
        enriched_row["_source_domain"] = source_domain
        enriched_row["_record_ref"] = record_ref

        enriched.append(enriched_row)

    return enriched


def _ordered_limited_records(df, order_candidates: list[str], limit: int) -> list[dict]:
    """
    Apply orderBy only for columns that exist, then limit, then convert to dicts.
    """
    order_cols = [c for c in order_candidates if c in df.columns]

    if order_cols:
        df = df.orderBy(*order_cols)

    return rows_to_dicts(df.limit(limit))


def _extract_priority_ts_params(ts_for_study) -> dict:
    """
    Extract priority TS parameters as a compact dict for LLM context.

    Returns a dict keyed by TSPARMCD. Each value is either:
      - a single string if only one distinct value exists for that param
      - a dict of {TSGRPID: TSVAL} if multiple groups exist, e.g. ACTSUB

    Only includes PRIORITY_TS_PARAMS. Deduplicates rows.
    """
    priority_df = (
        ts_for_study
        .where(F.col("TSPARMCD").isin(PRIORITY_TS_PARAMS))
        .select("TSPARMCD", "TSPARM", "TSGRPID", "TSVAL")
        .distinct()
        .orderBy("TSPARMCD", "TSGRPID")
    )

    rows = priority_df.collect()

    params = {}
    for row in rows:
        parmcd = row["TSPARMCD"]
        grpid = row["TSGRPID"]
        val = row["TSVAL"]

        if parmcd not in params:
            params[parmcd] = {"label": row["TSPARM"], "values": {}}

        group_key = grpid if grpid else "_default"
        params[parmcd]["values"][group_key] = val

    compact = {}
    for parmcd, info in params.items():
        values = info["values"]

        if len(values) == 1:
            compact[parmcd] = {
                "label": info["label"],
                "value": list(values.values())[0]
            }
        else:
            compact[parmcd] = {
                "label": info["label"],
                "values_by_group": values
            }

    return compact


def build_study_context_for_study(
    study_id: str,
    max_ts_tv_records_per_study: int = 200
) -> dict:
    """
    Build one compact SDTM-derived LLM context object for one study.

    This function is intentionally fact-sheet agnostic.
    It only describes what TS and TV can contribute.

    Source precedence across TS, TV, Fact Sheet, protocol template,
    and AI-generated content should be handled by the prompt or
    orchestration layer, not here.
    """

    # ---- TS ----
    ts_for_study = ts_df.where(F.col("STUDYID") == study_id)
    priority_ts = _extract_priority_ts_params(ts_for_study)

    # ---- TV ----
    tv_for_study = tv_df.where(F.col("STUDYID") == study_id)

    tv_records_raw = _ordered_limited_records(
        df=tv_for_study,
        order_candidates=["TVSEQ", "VISITNUM", "VISIT", "EPOCH"],
        limit=max_ts_tv_records_per_study
    )

    tv_records = add_record_ref(tv_records_raw, "SDTM TV")

    return {
        "study_id": study_id,
        "context_type": "sdtm_study_context",
        "structured_sources_available": ["SDTM TS", "SDTM TV"],
        "sdtm_context": {
            "priority_ts_params": priority_ts,
            "tv_records": tv_records
        },
        "source_usage": {
            "SDTM TS": {
                "purpose": (
                    "Trial-level metadata such as study type, phase, indication, "
                    "therapy, planned countries, age range, sex population, dates, "
                    "number of arms, and available subject disposition values."
                ),
                "priority_params": PRIORITY_TS_PARAMS,
                "may_inform_sections": TS_CONTEXT_SECTIONS
            },
            "SDTM TV": {
                "purpose": (
                    "Trial visit schedule, visit names, visit order, epochs, and "
                    "planned visit timing."
                ),
                "may_inform_sections": TV_CONTEXT_SECTIONS
            }
        },
        "generation_scope": {
            "protocol_type": "fictional ICH M11-style interventional clinical trial protocol draft",
            "sdtm_usage_guidance": [
                "Use SDTM TS for trial-level metadata when appropriate.",
                "Use SDTM TV for visit schedule and visit timing when appropriate.",
                "Do not assume SDTM TS or SDTM TV are the highest-priority sources if other source blocks are provided by the orchestration layer.",
                "Do not infer that objectives, endpoints, inclusion criteria, or exclusion criteria are unsupported solely because they are absent from TS or TV.",
                "Use the global prompt/source-precedence policy to decide which source wins when multiple source blocks conflict."
            ],
            "traceability_rule": (
                "Use source_domain = 'SDTM TS' for TS-derived values and "
                "source_domain = 'SDTM TV' for TV-derived visit schedule values."
            )
        },
        "generation_timestamp": datetime.now().isoformat()
    }


def get_study_ids() -> list[str]:
    """
    Return all STUDYID values found in TS or TV.
    """
    ts_study_ids = ts_df.select("STUDYID").where(F.col("STUDYID").isNotNull())
    tv_study_ids = tv_df.select("STUDYID").where(F.col("STUDYID").isNotNull())

    return [
        row["STUDYID"]
        for row in (
            ts_study_ids
            .unionByName(tv_study_ids)
            .distinct()
            .orderBy("STUDYID")
            .collect()
        )
    ]


def build_study_contexts(
    max_ts_tv_records_per_study: int = 200
) -> list[dict]:
    """
    Build a list of SDTM study contexts.

    One item = one study = one SDTM context block.
    """
    study_ids = get_study_ids()

    contexts = [
        build_study_context_for_study(
            study_id=study_id,
            max_ts_tv_records_per_study=max_ts_tv_records_per_study
        )
        for study_id in study_ids
    ]

    return contexts


ts_df = spark.read.table(f"{SDTM_SCHEMA}.{SDTM_TS_TABLE}")
tv_df = spark.read.table(f"{SDTM_SCHEMA}.{SDTM_TV_TABLE}")

# Filter to specific study IDs if specified
if STUDY_IDS:
    ts_df = ts_df.where(F.col("STUDYID").isin(STUDY_IDS))
    tv_df = tv_df.where(F.col("STUDYID").isin(STUDY_IDS))

study_contexts = build_study_contexts(max_ts_tv_records_per_study=10)

print(f"Built {len(study_contexts)} SDTM study context(s)")

for ctx in study_contexts:
    print(
        f"Study {ctx['study_id']}: "
        f"{len(ctx['sdtm_context']['priority_ts_params'])} priority TS params, "
        f"{len(ctx['sdtm_context']['tv_records'])} TV records"
    )

In [0]:
# Create table with auto-incrementing identity column if it doesn't exist
spark.sql("""
    CREATE TABLE IF NOT EXISTS m11_protocols (
        id BIGINT GENERATED ALWAYS AS IDENTITY,
        content VARIANT,
        run_time TIMESTAMP,
        tokens INT,
        model STRING,
        mlflow_run_id STRING
    )
""")

In [0]:
import json
import hashlib
import time
from pyspark.sql.functions import lit, current_timestamp, expr
from databricks.sdk import WorkspaceClient
from openai import OpenAI
import mlflow

# --- MLflow tracing setup ---
mlflow.openai.autolog()
EXPERIMENT_PATH = "/Users/hfsl@novonordisk.com/ich-m11-protocol-gen/mlflow_ich_m11"
mlflow.set_experiment(EXPERIMENT_PATH)

# --- Load schema ---
schema_path = "assets/schema_m11_medium.json"
with open(schema_path) as f:
    schema_raw = f.read()
m11_schema = json.loads(schema_raw)
schema_checksum = hashlib.sha256(schema_raw.encode()).hexdigest()[:16]

# --- Prompt version fingerprint ---
prompt_fingerprint = hashlib.sha256(
    (system_message + user_message_template).encode()
).hexdigest()[:16]

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()
mlflow_client = mlflow.MlflowClient()

# --- Model configuration ---
# Change this to compare different models (e.g., "databricks-claude-sonnet-4", "databricks-gpt-5-4-mini")
model = "databricks-gpt-5-4-mini"

run_time = datetime.now()
protocol_responses = []
trace_records = []  # Store trace IDs + metadata for Phase 2 judge evaluation

# --- Skip studies with existing PDFs if overwrite is disabled ---
if not PROTOCOL_OVERWRITE:
    existing_pdfs = {f.name for f in dbutils.fs.ls(vol_path) if f.name.endswith(".pdf")}
    original_count = len(study_contexts)
    study_contexts = [
        ctx for ctx in study_contexts
        if f"PROTOCOL_{ctx['study_id']}.pdf" not in existing_pdfs
    ]
    skipped = original_count - len(study_contexts)
    if skipped > 0:
        print(f"⏭️ Skipping {skipped} study(ies) with existing PDFs (PROTOCOL_OVERWRITE=False)")

if not study_contexts:
    print("✅ All protocols already exist. Nothing to generate. Set PROTOCOL_OVERWRITE=True to regenerate.")
    run = None
else:
  print(f"🔎 Generating protocol drafts for {len(study_contexts)} study context(s) using model: {model}")
  with mlflow.start_run(run_name=f"ich_m11_{model}_{run_time:%Y%m%d_%H%M}") as run:
    # Log pipeline-level params
    mlflow.log_params({
        "model": model,
        "schema_file": schema_path,
        "schema_checksum": schema_checksum,
        "prompt_fingerprint": prompt_fingerprint,
        "num_studies": len(study_contexts),
        "study_ids": ",".join(ctx["study_id"] for ctx in study_contexts),
    })

    total_input_tokens = 0
    total_output_tokens = 0

    for i, ctx in enumerate(study_contexts, 1):
        study_id = ctx.get("study_id", "UNKNOWN")
        print(f"⏳ [{i}/{len(study_contexts)}] Generating protocol for study: {study_id} ...")
     

        # Use fact sheet if present, otherwise "None"
        study_fact_sheet = study_fact_sheets.get(study_id, None)   
        # log factsheet content
        if study_fact_sheet:
            print(f"✅ Using fact sheet for study {study_id}")
        else:
            print(f"⚠️ No fact sheet for study {study_id}!")

        # --- Phase 1: Prompt generation ---
        user_message = user_message_template.format(
            study_context_json=ctx,
            study_fact_sheet=study_fact_sheet
        )

            

        t0 = time.time()
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "m11_protocol",
                    "schema": m11_schema,
                    "strict": True,
                },
            }
        )
        latency_s = time.time() - t0

        # --- Phase 1: Enhanced tracing — tag each trace with per-study metadata ---
        trace_id = mlflow.get_last_active_trace_id()
        if trace_id:
            mlflow_client.set_trace_tag(trace_id, "study_id", study_id)
            mlflow_client.set_trace_tag(trace_id, "model", model)
            mlflow_client.set_trace_tag(trace_id, "prompt_fingerprint", prompt_fingerprint)
            mlflow_client.set_trace_tag(trace_id, "schema_checksum", schema_checksum)
            mlflow_client.set_trace_tag(trace_id, "mlflow.run_id", run.info.run_id)
            trace_records.append({
                "trace_id": trace_id,
                "study_id": study_id,
                "model": model,
                "latency_s": latency_s,
            })
            print(f"   📍 Trace tagged: {trace_id}")

        finish_reason = response.choices[0].finish_reason
        usage = response.usage
        if finish_reason == "length":
            print(f"⚠️ Study {study_id}: Output was truncated due to max_tokens limit.")
        print(f"✅ Study {study_id}: Finish reason: {finish_reason}, Tokens used: {usage.total_tokens}, Latency: {latency_s:.1f}s")

        raw_content = response.choices[0].message.content
        parsed = json.loads(raw_content) if isinstance(raw_content, str) else raw_content
        protocol_responses.append(parsed)

        # Per-study metrics
        proto = parsed.get("protocol", parsed)
        num_sections = len(proto.get("protocol_sections", []))
        avg_narrative = (
            sum(len(s.get("narrative_text", "")) for s in proto.get("protocol_sections", []))
            / max(num_sections, 1)
        )

        total_input_tokens += usage.prompt_tokens
        total_output_tokens += usage.completion_tokens

        mlflow.log_metrics({
            f"{study_id}/input_tokens": usage.prompt_tokens,
            f"{study_id}/output_tokens": usage.completion_tokens,
            f"{study_id}/total_tokens": usage.total_tokens,
            f"{study_id}/latency_s": round(latency_s, 2),
            f"{study_id}/num_sections": num_sections,
            f"{study_id}/avg_narrative_chars": round(avg_narrative),
            f"{study_id}/truncated": 1 if finish_reason == "length" else 0,
        }, step=i)

    # Aggregate metrics
    mlflow.log_metrics({
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "total_tokens": total_input_tokens + total_output_tokens,
    })

    # Log schema file as artifact
    mlflow.log_artifact(schema_path)

    print(f"\n✅ Protocol draft generation complete for {len(protocol_responses)} study context(s)")
    print(f"📊 MLflow run: {run.info.run_id}")
    print(f"📍 {len(trace_records)} trace(s) tagged for judge evaluation")
    print(f"\n💡 To evaluate quality, run the judge notebook: ich_m11_quality_judge")


In [0]:
# Save results to table — the identity column is auto-populated on append
content_json = json.dumps(protocol_responses)
df = spark.createDataFrame([(content_json,)], ["content"])  
df = df.withColumn("content", expr("parse_json(content)"))
df = df.withColumn("run_time", current_timestamp())
df = df.withColumn("tokens", lit(response.usage.total_tokens))
df = df.withColumn("model", lit(model))
df = df.withColumn("mlflow_run_id", lit(run.info.run_id))
display(df)
if SAVE_AS_DELTA:
    df.write.mode("append").saveAsTable("m11_protocols")


In [0]:
if SAVE_AS_DELTA:
    row = spark.sql("""
        SELECT 
            id,
            collect_list(c.value:protocol) as protocols,
            collect_list(c.value:protocol:title_page:sponsor_protocol_identifier::string) AS protocol_ids
        FROM m11_protocols p, LATERAL  variant_explode(p.content) AS c
        WHERE id = (SELECT max(id) FROM m11_protocols) GROUP BY id
        ORDER BY id DESC
    """).collect()

if len(row) > 0:
    protocol_ids = row[0]['protocol_ids']
    protocols = row[0]['protocols']
    print(f"🔍 Found {len(protocols)} protocol(s) 📝")
    for i, protocol_id in enumerate(protocol_ids):
        print(f"📄 Protocol {i+1}: study_id = {protocol_id} 🧬")
else:
    print("❌ No protocols found in table 🚫")

In [0]:
import importlib
import ich_m11_renderer
importlib.reload(ich_m11_renderer)

from ich_m11_renderer import protocol_to_html_body, render_html_document, html_to_pdf

DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl")

l_preview_url = lambda vol_path, filename :  (
        f"https://{DATABRICKS_HOST}/explore/data{vol_path}"
        f"?filePreviewPath={filename}"
    )

# Set to true to save intermediate HTML files; use for debugging
SAVE_HTML = False

for protocol_variant_val in protocols:
    protocol = protocol_variant_val.toPython()
    study_id = protocol["title_page"]["sponsor_protocol_identifier"]
    print(f"🖨️ Generating PDF for study {study_id} ...")
  

    # Render protocol JSON directly to HTML body
    body_html = protocol_to_html_body(protocol)

    # Wrap body HTML with CSS + template
    protocol_html = render_html_document(
        body_html=body_html,
        css_path="assets/ich_m11.css",
        template_path="assets/ich_m11_template.html",
        document_title="ICH M11 Protocol Draft",
    )

    # Save intermediate HTML for debugging
    if SAVE_HTML:
        html_file_name = f"PROTOCOL_{study_id}.html"
        output_html_path = f"{vol_path}/{html_file_name}"
        with open(output_html_path, "w", encoding="utf-8") as f:
            f.write(protocol_html)
        print(f"Written {study_id}.html {l_preview_url(vol_path,html_file_name)}")

    # Convert in-memory HTML directly to PDF
    protocol_file_name = f"PROTOCOL_{study_id}.pdf"
    output_pdf_path = f"{vol_path}/{protocol_file_name}"
    html_to_pdf(
        output_pdf_path=output_pdf_path,
        html_string=protocol_html,
    )

    # Preserve Databricks file preview URL
    preview_url = l_preview_url(vol_path, protocol_file_name)
    print(f"Written {study_id}.pdf {preview_url}")